# Notebook 4: Model Comparison — TFT vs TiDE
**Phase 5 of the hackathon plan**

Goal: Load both sets of validation scores, build a side-by-side comparison table, identify the winner, and document the reasons clearly for the presentation.

## Step 1 — Setup & Load Both Score Files

In [ ]:
import sys
sys.path.append("../03_scripts")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from utils import OUTPUT_TFT, OUTPUT_TIDE, OUTPUT_FINAL

%matplotlib inline

# Load validation scores from both models
with open(OUTPUT_TFT  / "validation_scores.json") as f:
    tft_scores = json.load(f)
with open(OUTPUT_TIDE / "validation_scores.json") as f:
    tide_scores = json.load(f)

# TM1 baseline (from the brief)
tm1_scores = {"WAPE (overall)": 0.137, "MACRO-WAPE (by brand)": None, "sMAPE": None, "Bias (avg)": None}

print("TFT scores:",  tft_scores)
print("TiDE scores:", tide_scores)

## Step 2 — Side-by-Side Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "TM1 Baseline (old)": tm1_scores,
    "TFT":               tft_scores,
    "TiDE":              tide_scores,
}).T

print("=== MODEL COMPARISON ===")
print(comparison.to_string())

# Highlight winner
best_wape = comparison["WAPE (overall)"].astype(float).idxmin()
print(f"
Winner (lowest WAPE): {best_wape}")
comparison.to_csv(OUTPUT_FINAL / "model_comparison.csv")
print("Saved to 04_outputs/final/model_comparison.csv")

## Step 3 — Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = ["TM1 Baseline", "TFT", "TiDE"]
wapes  = [
    float(tm1_scores["WAPE (overall)"]),
    float(tft_scores["WAPE (overall)"]),
    float(tide_scores["WAPE (overall)"]),
]
colors = ["#d9534f", "#5bc0de", "#5cb85c"]

# Bar chart: overall WAPE
axes[0].bar(models, wapes, color=colors)
axes[0].set_title("Overall WAPE (lower = better)")
axes[0].set_ylabel("WAPE")
axes[0].axhline(y=0.137, color="red", linestyle="--", label="TM1 bar")
for i, v in enumerate(wapes):
    axes[0].text(i, v + 0.002, f"{v:.4f}", ha="center", fontweight="bold")

# Improvement over baseline
improvement = [(0.137 - w) / 0.137 * 100 for w in wapes[1:]]
axes[1].bar(["TFT", "TiDE"], improvement, color=["#5bc0de", "#5cb85c"])
axes[1].set_title("% Improvement over TM1 Baseline")
axes[1].set_ylabel("% Reduction in Error")
for i, v in enumerate(improvement):
    axes[1].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontweight="bold")

# Winner announcement
winner_color = "#5cb85c" if best_wape == "TiDE" else "#5bc0de"
axes[2].text(0.5, 0.55, f"Winner:", ha="center", fontsize=14, color="gray", transform=axes[2].transAxes)
axes[2].text(0.5, 0.45, best_wape, ha="center", fontsize=28, fontweight="bold",
             color=winner_color, transform=axes[2].transAxes)
axes[2].axis("off")
axes[2].set_title("Selected Model")

plt.tight_layout()
plt.savefig("../06_presentation/charts/04_model_comparison.png", dpi=150)
plt.show()
print("Chart saved.")

## Step 4 — Why We Chose the Winner

This section is filled in after running both models. It will be copied into the presentation.

In [ ]:
winner = best_wape  # auto-set from above

reasoning = f"""
FINALIZED MODEL: {winner}

Reason 1 — Accuracy
  {winner} achieved a WAPE of {min(tft_scores["WAPE (overall)"], tide_scores["WAPE (overall)"]):.4f}
  vs the company baseline of 0.137 — a {(0.137 - min(tft_scores["WAPE (overall)"], tide_scores["WAPE (overall)"])) / 0.137 * 100:.1f}% improvement.

Reason 2 — Architecture fit
  TiDE was purpose-built with a direct highway for future-known variables
  (insurance coverage, drug prices, promotions), which are our strongest signals.
  TFT achieves the same through attention, but TiDE's simpler design reduces overfitting.

Reason 3 — Consistency across brands
  [Fill in after seeing MACRO-WAPE results: which model was more balanced
   across all 8 drugs, including newly launched ones?]

Reason 4 — Speed
  TiDE trains approximately 5-10x faster than TFT, allowing more
  iterations and tuning within the hackathon timeframe.
"""

print(reasoning)

# Save the reasoning
with open(OUTPUT_FINAL / "model_selection_rationale.txt", "w") as f:
    f.write(reasoning)
print("Saved to 04_outputs/final/model_selection_rationale.txt")
print("
Phase 5 complete. Proceed to Notebook 05 (MinTrace).")